### Primero debemos instalar las librerias necesarias para generar el web scrapping, hemos intentado con request pero no hemos podido por eso usamos Selenium que es mas potente. Utilizamos Selenium para acceder a la web como si fuera un navegador, BeautifulSoup para analizar el HTML y Pandas para organizar los datos en forma de tabla.

In [1]:
!pip install selenium
!apt-get update
!apt-get install -y chromium-browser
!apt-get install -y chromium-chromedriver
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

zsh:1: command not found: apt-get
zsh:1: command not found: apt-get
zsh:1: command not found: apt-get


### Una vez instaladas nuestras librerias procedemos a intentar hacer un request a la página de F1, tambien configuramos el navegador (Chrome) con algunas opciones para evitar problemas y simular un comportamiento más "humano". Luego inicializamos el driver de Selenium (abre el navegador automáticamente)

In [ ]:


options = Options()

options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

year=2025

url = f"https://lat.motorsport.com/f1/drivers/?y={year}"

driver.get(url)

time.sleep(5)

print(driver.title)

html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

print(len(html))



Fórmula 1 Pilotos - Todos Fórmula 1 Drivers
745286


### Una vez generado el acceso podemos importar beautiful soup para poder encontrar los anuncios, para esto dentro de la pagina inspeccionamos el HTML y encontramos cuales son las clases que dividen a los pilotos.

In [4]:
pilotos = soup.find_all("a", class_="ms-item--driver ms-item-vert-d ms-item-vert-t ms-item-vert-m")

print(len(pilotos))

21


### Probamos imprimir los datos del primer piloto de la lista.

In [5]:
primer = pilotos[0]

print(primer.text)












1


            Max Verstappen        


Equipo:
                Red Bull Racing            

Fecha de nacimiento:
                1997-09-30                (Edad 28)


Nacionalidad:
                Países Bajos, Holanda            





### Una vez que sabemos que podemos acceder a todos los datos, procedemos a ver si podemos encontrar todos los nombres de los pilotos, para esto usamos Beautiful Soup con la función "find_all". 

In [6]:
nombres_pilotos = soup.find_all("p", class_="ms-item__title")

print(nombres_pilotos)

[<p class="ms-item__title">
            Max Verstappen        </p>, <p class="ms-item__title">
            Lando Norris        </p>, <p class="ms-item__title">
            Gabriel  Bortoleto        </p>, <p class="ms-item__title">
            Isack Hadjar        </p>, <p class="ms-item__title">
            Jack Doohan        </p>, <p class="ms-item__title">
            Pierre Gasly        </p>, <p class="ms-item__title">
            Andrea Kimi Antonelli        </p>, <p class="ms-item__title">
            Fernando Alonso        </p>, <p class="ms-item__title">
            Charles Leclerc        </p>, <p class="ms-item__title">
            Lance Stroll        </p>, <p class="ms-item__title">
            Yuki Tsunoda        </p>, <p class="ms-item__title">
            Alex Albon        </p>, <p class="ms-item__title">
            Nico Hulkenberg        </p>, <p class="ms-item__title">
            Liam Lawson        </p>, <p class="ms-item__title">
            Esteban Ocon        </p>, <p

### Solo a modo de confirmacion imprimimos el nombre del primer piloto (para saber si podemos dejar la informacion de nombre en un formato que nos sirva para el dataset)

In [7]:

print(nombres_pilotos[0].text.strip())



Max Verstappen


### Siguiendo con la información del mismo piloto, para poder extraer la información de la nacionalidad buscamos por la clase, a su vez utilizamos la palabra "Nacionalidad" porque se repite cada vez en el HTML. Una vez extraida esta data pasamos a hacer el strip para conseguir el dato limpio.

In [8]:
primer = pilotos[0]

pais = [
    row.get_text(strip=True)
    for row in primer.find_all("div", class_="ms-item-driver__info-row") 
    if "Nacionalidad" in row.text][0]

pais = pais.replace("Nacionalidad:", "").strip()

print(pais)

Países Bajos, Holanda


###  Para obtener la fecha de nacimiento procedemos a hacer un proceso similar al anterior, obviamente es otra clase y tambien cambia el texto, en este caso "Fecha de nacimiento". Utilizando solamente la primer parte del codigo extrae el texto pero viene acompañado de la edad lo cual no necesitamos ya que se refiere a la edad en ese momento y tampoco es dificil de calcular si tenemos la fecha de nacimiento por lo que luego usamos split y strip para obtener el dato limpio.

In [9]:
primer = pilotos[0]

fecha_nacimiento = [
    row.get_text(strip=True)
    for row in primer.find_all("div", class_="ms-item-driver__info-row")
    if "Fecha de nacimiento" in row.text][0]

fecha_nacimiento = fecha_nacimiento.replace("Fecha de nacimiento:", "").split("(")[0].strip()

print(fecha_nacimiento)

1997-09-30


### A la hoar de obtener el dato del equipo, es decir, para que escuderia corre cada piloto, debemos hacer un proceso muy similar a los anteriores obviamente cambiando la clase y el texto que lo precede.

In [10]:
equipo = [
    row.get_text(strip=True)
    for row in primer.find_all("div", class_="ms-item-driver__info-row")
    if "Equipo" in row.text][0]

team = equipo.replace("Equipo:", "").strip()

print(team)

Red Bull Racing


### Una vez que pudimos obtener toda la informacion de un corredor para un año procedemos a iterar por cada piloto y cada año. Para cambiar el año debemos cambiarlo en el link, por lo que debemos generar un link dinamico por el cual podamos iterar los años que querramos usar, en este caso elegimos los ultimos 10 años. Es importante primero iterar por los años con el for loop y luego dentro de cada año utilizamos otro for para iterar por cada piloto. Luego dentro de cada piloto utilizamos otro for loop para obtener cada dato del mismo y una vez que tengamos todos los datos del piloto usamos "append" para añadirlo a nuestra tabla (anteriormente creada y en blanco), por ultimo lo convertirmos en un data frame.

In [17]:
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

data_total = []

for year in years:
    
    url = f"https://lat.motorsport.com/f1/drivers/?y={year}"
    
    driver.get(url)
    time.sleep(5)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    pilotos = soup.find_all("a", class_="ms-item--driver ms-item-vert-d ms-item-vert-t ms-item-vert-m")
    
    for piloto in pilotos:
        
        nombre = piloto.find("p", class_="ms-item__title").text.strip()
        
        pais = None
        fecha = None
        equipo = None
        
        rows = piloto.find_all("div", class_="ms-item-driver__info-row")
        
        for row in rows:
            texto = row.get_text(strip=True)
            
            if "Nacionalidad" in texto:
                pais = texto.replace("Nacionalidad:", "").strip()
            
            if "Fecha de nacimiento" in texto:
                fecha = texto.replace("Fecha de nacimiento:", "").split("(")[0].strip()
            
            if "Equipo" in texto:
                equipo = texto.replace("Equipo:", "").strip()
        
        data_total.append({
            "nombre": nombre,
            "pais": pais,
            "fecha_nacimiento": fecha,
            "equipo": equipo,
            "year": year
        })

df = pd.DataFrame(data_total)

print(df)

                nombre         pais fecha_nacimiento           equipo  year
0         Lance Stroll       Canadá       1998-10-29         Williams  2017
1    Stoffel Vandoorne      Bélgica       1992-03-26          McLaren  2017
2     Daniel Ricciardo    Australia       1989-07-01  Red Bull Racing  2017
3     Sebastian Vettel     Alemania       1987-07-03          Ferrari  2017
4       Kimi Raikkonen    Finlandia       1979-10-17          Ferrari  2017
..                 ...          ...              ...              ...   ...
207       Carlos Sainz       España       1994-09-01         Williams  2026
208     George Russell  Reino Unido       1998-02-15         Mercedes  2026
209    Valtteri Bottas    Finlandia       1989-08-28         Cadillac  2026
210      Oscar Piastri    Australia       2001-04-06          McLaren  2026
211     Oliver Bearman  Reino Unido       2005-05-08          Haas F1  2026

[212 rows x 5 columns]


### Una vez creado nuestro data frame pasamos a corroborar que este bien creado sin errores. Vemos su forma "shape", imprimimos las primero 5 lineas y las ultimas 5.

In [18]:
print(df.shape)
print(df.head())
print(df.tail())

(212, 5)
              nombre       pais fecha_nacimiento           equipo  year
0       Lance Stroll     Canadá       1998-10-29         Williams  2017
1  Stoffel Vandoorne    Bélgica       1992-03-26          McLaren  2017
2   Daniel Ricciardo  Australia       1989-07-01  Red Bull Racing  2017
3   Sebastian Vettel   Alemania       1987-07-03          Ferrari  2017
4     Kimi Raikkonen  Finlandia       1979-10-17          Ferrari  2017
              nombre         pais fecha_nacimiento    equipo  year
207     Carlos Sainz       España       1994-09-01  Williams  2026
208   George Russell  Reino Unido       1998-02-15  Mercedes  2026
209  Valtteri Bottas    Finlandia       1989-08-28  Cadillac  2026
210    Oscar Piastri    Australia       2001-04-06   McLaren  2026
211   Oliver Bearman  Reino Unido       2005-05-08   Haas F1  2026


### Corroboramos que no haya valores nulos.

In [19]:
print(df.isnull().sum())

nombre              0
pais                0
fecha_nacimiento    0
equipo              0
year                0
dtype: int64


### Acomodamos el data frame ordenandolo por año y por nombre, imprimimos las primeros 5 lineas para corroborar.

In [21]:
df = df.sort_values(by=["year", "nombre"])
df.head()

,nombre,pais,fecha_nacimiento,equipo,year
18,Antonio Giovinazzi,Italia,1993-12-14,Sauber F1 Team,2017
14,Brendon Hartley,Nueva Zelanda,1989-11-10,RB,2017
20,Carlos Sainz,España,1994-09-01,RB,2017
21,Carlos Sainz,España,1994-09-01,Renault F1 Team,2017
2,Daniel Ricciardo,Australia,1989-07-01,Red Bull Racing,2017


### Una vez que estamos satisfechos con el resultado, procedemos a guardar nuestro data set en un archivo csv.

In [22]:
df.to_csv("f1_drivers_dataset.csv", index=False)